# K-means Clustering 

In this this exercise, you will implement the K-means algorithm and use it for image compression. 

* You will start with a sample dataset that will help you gain an intuition of how the K-means algorithm works. 
* After that, you wil use the K-means algorithm for image compression by reducing the number of colors that occur in an image to only those that are most common in that image.




# Outline
- [ 1 - Implementing K-means](#1)
  - [ 1.1 Finding closest centroids](#1.1)
    - [ Exercise 1](#ex01)
  - [ 1.2 Computing centroid means](#1.2)
    - [ Exercise 2](#ex02)
- [ 2 - K-means on a sample dataset ](#2)
- [ 3 - Random initialization](#3)
- [ 4 - Image compression with K-means](#4)
  - [ 4.1 Dataset](#4.1)
  - [ 4.2 K-Means on image pixels](#4.2)
  - [ 4.3 Compress the image](#4.3)


# K-means 聚类

在本次练习中，你将实现 K-means 算法并将其用于图像压缩。

* 你将先从一个示例数据集入手，帮助你直观理解 K-means 算法的工作原理。
* 之后，你将使用 K-means 算法进行图像压缩，通过将图像中出现的颜色数量减少为其中最常见的少数几种颜色来实现压缩。




# 目录
- [ 1 - 实现 K-means](#1)
  - [ 1.1 寻找最近的聚类中心](#1.1)
    - [ 练习 1](#ex01)
  - [ 1.2 计算聚类中心均值](#1.2)
    - [ 练习 2](#ex02)
- [ 2 - 在示例数据集上运行 K-means ](#2)
- [ 3 - 随机初始化](#3)
- [ 4 - 使用 K-means 进行图像压缩](#4)
  - [ 4.1 数据集](#4.1)
  - [ 4.2 对图像像素运行 K-Means](#4.2)
  - [ 4.3 压缩图像](#4.3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils import *

%matplotlib inline

<a name="1"></a>
## 1 - Implementing K-means

The K-means algorithm is a method to automatically cluster similar
data points together. 

* Concretely, you are given a training set $\{x^{(1)}, ..., x^{(m)}\}$, and you want
to group the data into a few cohesive “clusters”. 


* K-means is an iterative procedure that
     * Starts by guessing the initial centroids, and then 
     * Refines this guess by 
         * Repeatedly assigning examples to their closest centroids, and then 
         * Recomputing the centroids based on the assignments.
         

* In pseudocode, the K-means algorithm is as follows:

    ``` python
    # Initialize centroids
    # K is the number of clusters
    centroids = kMeans_init_centroids(X, K)
    
    for iter in range(iterations):
        # Cluster assignment step: 
        # Assign each data point to the closest centroid. 
        # idx[i] corresponds to the index of the centroid 
        # assigned to example i
        idx = find_closest_centroids(X, centroids)

        # Move centroid step: 
        # Compute means based on centroid assignments
        centroids = compute_means(X, idx, K)
    ```


* The inner-loop of the algorithm repeatedly carries out two steps: 
    * (i) Assigning each training example $x^{(i)}$ to its closest centroid, and
    * (ii) Recomputing the mean of each centroid using the points assigned to it. 
    
    
* The $K$-means algorithm will always converge to some final set of means for the centroids. 

* However, that the converged solution may not always be ideal and depends on the initial setting of the centroids.
    * Therefore, in practice the K-means algorithm is usually run a few times with different random initializations. 
    * One way to choose between these different solutions from different random initializations is to choose the one with the lowest cost function value (distortion).

You will implement the two phases of the K-means algorithm separately
in the next sections. 
* You will start by completing `find_closest_centroid` and then proceed to complete `compute_centroids`.

## 1 - 实现 K-means

K-means 算法是一种能够自动将相似数据点聚在一起的方法。

* 具体来说，给定一个训练集 $\{x^{(1)}, ..., x^{(m)}\}$，你希望将数据划分为若干个紧密的“簇（clusters）”。


* K-means 是一个迭代过程：
     * 首先猜测初始的聚类中心（质心），然后
     * 通过以下步骤不断优化该猜测：
         * 反复将每个样本分配给距离它最近的聚类中心，然后
         * 根据分配结果重新计算聚类中心。
         

* 用伪代码表示，K-means 算法如下：

    ``` python
    # 初始化聚类中心
    # K 是簇的数量
    centroids = kMeans_init_centroids(X, K)
    
    for iter in range(iterations):
        # 簇分配步骤：
        # 将每个数据点分配给最近的聚类中心。
        # idx[i] 对应于分配给样本 i 的聚类中心索引
        idx = find_closest_centroids(X, centroids)

        # 移动聚类中心步骤：
        # 根据簇分配结果计算均值
        centroids = compute_means(X, idx, K)
    ```


* 算法的内层循环反复执行以下两个步骤：
    * (i) 将每个训练样本 $x^{(i)}$ 分配给距离它最近的聚类中心，以及
    * (ii) 使用分配给每个聚类中心的所有点，重新计算该聚类中心的均值。
    
    
* $K$-means 算法总会收敛到某组最终的聚类中心均值。

* 然而，收敛得到的解并不总是理想的，它取决于聚类中心的初始设置。
    * 因此，在实践中通常会使用不同的随机初始化多次运行 K-means 算法。
    * 在不同随机初始化得到的多个解中进行选择的一种方法是：挑选代价函数值（失真度，distortion）最低的那个解。

在接下来的章节中，你将分别实现 K-means 算法的两个阶段。
* 你将先补全 `find_closest_centroid`，然后再补全 `compute_centroids`。

<a name="1.1"></a>
### 1.1 Finding closest centroids

In the “cluster assignment” phase of the K-means algorithm, the
algorithm assigns every training example $x^{(i)}$ to its closest
centroid, given the current positions of centroids. 

<a name="ex01"></a>
### Exercise 1

Your task is to complete the code in `find_closest_centroids`. 
* This function takes the data matrix `X` and the locations of all
centroids inside `centroids` 
* It should output a one-dimensional array `idx` (which has the same number of elements as `X`) that holds the index  of the closest centroid (a value in $\{1,...,K\}$, where $K$ is total number of centroids) to every training example .
* Specifically, for every example $x^{(i)}$ we set
$$c^{(i)} := j \quad \mathrm{that \; minimizes} \quad ||x^{(i)} - \mu_j||^2,$$
where 
 * $c^{(i)}$ is the index of the centroid that is closest to $x^{(i)}$ (corresponds to `idx[i]` in the starter code), and 
 * $\mu_j$ is the position (value) of the $j$’th centroid. (stored in `centroids` in the starter code)
 
If you get stuck, you can check out the hints presented after the cell below to help you with the implementation.

### 1.1 寻找最近的聚类中心

在 K-means 算法的“簇分配”阶段，算法会根据当前聚类中心的位置，将每个训练样本 $x^{(i)}$ 分配给距离它最近的聚类中心。

### 练习 1

你的任务是补全 `find_closest_centroids` 中的代码。
* 该函数接收数据矩阵 `X` 和包含所有聚类中心位置的 `centroids`。
* 它应输出一个一维数组 `idx`（元素个数与 `X` 的样本数相同），其中保存了距离每个训练样本最近的聚类中心的索引（取值为 $\{1,...,K\}$，其中 $K$ 为聚类中心总数；在 Python 中为 0 到 $K-1$）。
* 具体而言，对于每个样本 $x^{(i)}$，我们设定：
$$c^{(i)} := j \quad \mathrm{that \; minimizes} \quad ||x^{(i)} - \mu_j||^2,$$
其中：
 * $c^{(i)}$ 是距离 $x^{(i)}$ 最近的聚类中心的索引（对应初始代码中的 `idx[i]`），
 * $\mu_j$ 是第 $j$ 个聚类中心的位置（值）（保存在初始代码的 `centroids` 中）。
 
如果遇到困难，你可以查看下方单元格之后的提示来帮助你完成实现。

In [ ]:
# UNQ_C1
# GRADED FUNCTION: find_closest_centroids

def find_closest_centroids(X, centroids):
    """
    Computes the centroid memberships for every example
    
    Args:
        X (ndarray): (m, n) Input values      
        centroids (ndarray): k centroids
    
    Returns:
        idx (array_like): (m,) closest centroids
    
    """

    # Set K
    K = centroids.shape[0]

    # You need to return the following variables correctly
    idx = np.zeros(X.shape[0], dtype=int)

    ### START CODE HERE ###

    ### END CODE HERE ###
    
    return idx

<details>
  <summary><font size="3" color="darkgreen"><b>Click for hints</b></font></summary>
    
    
* Here's how you can structure the overall implementation for this function
    ```python 
    def find_closest_centroids(X, centroids):
    
        # Set K
        K = centroids.shape[0]
    
        # You need to return the following variables correctly
        idx = np.zeros(X.shape[0], dtype=int)
    
        ### START CODE HERE ###
        for i in range(X.shape[0]):
            # Array to hold distance between X[i] and each centroids[j]
            distance = [] 
            for j in range(centroids.shape[0]):
                norm_ij = # Your code to calculate the norm between (X[i] - centroids[j])
                distance.append(norm_ij)
            
            idx[i] = # Your code here to calculate index of minimum value in distance
        ### END CODE HERE ###
        return idx
    ```
  
    If you're still stuck, you can check the hints presented below to figure out how to calculate `norm_ij` and `idx[i]`.
    
    <details>
          <summary><font size="2" color="darkblue"><b>Hint to calculate norm_ij</b></font></summary>
           &emsp; &emsp; You can use <a href="https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html">np.linalg.norm</a> to calculate the norm 
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; More hints to calculate norm_ij</b></font></summary>
               &emsp; &emsp; You can compute norm_ij as <code>norm_ij = np.linalg.norm(X[i] - centroids[j]) </code>
           </details>
    </details>

     <details>
          <summary><font size="2" color="darkblue"><b>Hint to calculate idx[i]</b></font></summary>
          &emsp; &emsp; You can use <a href="https://numpy.org/doc/stable/reference/generated/numpy.argmin.html">np.argmin</a> to find the index of the minimum value
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; More hints to calculate idx[i]</b></font></summary>
              &emsp; &emsp; You can compute idx[i] as <code>idx[i] = np.argmin(distance)</code>
          </details>
    </details>
        
    </details>

</details>

    


<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
    
* 以下是该函数的整体实现结构：
    ```python 
    def find_closest_centroids(X, centroids):
    
        # 设置 K
        K = centroids.shape[0]
    
        # 你需要正确返回以下变量
        idx = np.zeros(X.shape[0], dtype=int)
    
        ### START CODE HERE ###
        for i in range(X.shape[0]):
            # 用于保存 X[i] 与每个 centroids[j] 之间距离的列表
            distance = [] 
            for j in range(centroids.shape[0]):
                norm_ij = # 在此处编写代码，计算 (X[i] - centroids[j]) 的范数
                distance.append(norm_ij)
            
            idx[i] = # 在此处编写代码，计算 distance 中最小值的索引
        ### END CODE HERE ###
        return idx
    ```
  
    如果你仍然卡住了，可以查看下面的提示来了解如何计算 `norm_ij` 和 `idx[i]`。
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 norm_ij 的提示</b></font></summary>
           &emsp; &emsp; 你可以使用 <a href="https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html">np.linalg.norm</a> 来计算范数
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 关于计算 norm_ij 的更多提示</b></font></summary>
               &emsp; &emsp; 你可以这样计算 norm_ij：<code>norm_ij = np.linalg.norm(X[i] - centroids[j]) </code>
           </details>
    </details>

     <details>
          <summary><font size="2" color="darkblue"><b>计算 idx[i] 的提示</b></font></summary>
          &emsp; &emsp; 你可以使用 <a href="https://numpy.org/doc/stable/reference/generated/numpy.argmin.html">np.argmin</a> 来查找最小值的索引
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 关于计算 idx[i] 的更多提示</b></font></summary>
              &emsp; &emsp; 你可以这样计算 idx[i]：<code>idx[i] = np.argmin(distance)</code>
          </details>
    </details>
        
    </details>

</details>

Now let's check your implementation using an example dataset

现在让我们使用一个示例数据集来检查你的实现。

In [ ]:
# Load an example dataset that we will be using
X = load_data()

The code below prints the first five elements in the variable `X` and the dimensions of the variable

下面的代码打印了变量 `X` 中的前五个元素以及该变量的维度。

In [ ]:
print("First five elements of X are:\n", X[:5]) 
print('The shape of X is:', X.shape)

In [ ]:
# Select an initial set of centroids (3 Centroids)
initial_centroids = np.array([[3,3], [6,2], [8,5]])

# Find closest centroids using initial_centroids
idx = find_closest_centroids(X, initial_centroids)

# Print closest centroids for the first three elements
print("First three elements in idx are:", idx[:3])

# UNIT TEST
from public_tests import *

find_closest_centroids_test(find_closest_centroids)


**Expected Output**:
<table>
  <tr>
    <td> <b>First three elements in idx are<b></td>
    <td> [0 2 1] </td> 
  </tr>
</table>

**预期输出**：
<table>
  <tr>
    <td> <b>First three elements in idx are<b></td>
    <td> [0 2 1] </td> 
  </tr>
</table>

<a name="1.2"></a>
### 1.2 Computing centroid means

Given assignments of every point to a centroid, the second phase of the
algorithm recomputes, for each centroid, the mean of the points that
were assigned to it.


<a name="ex02"></a>
### Exercise 2

Please complete the `compute_centroids` below to recompute the value for each centroid

* Specifically, for every centroid $\mu_k$ we set
$$\mu_k = \frac{1}{|C_k|} \sum_{i \in C_k} x^{(i)}$$ 

    where 
    * $C_k$ is the set of examples that are assigned to centroid $k$
    * $|C_k|$ is the number of examples in the set $C_k$


* Concretely, if two examples say $x^{(3)}$ and $x^{(5)}$ are assigned to centroid $k=2$,
then you should update $\mu_2 = \frac{1}{2}(x^{(3)}+x^{(5)})$.

If you get stuck, you can check out the hints presented after the cell below to help you with the implementation.

### 1.2 计算聚类中心均值

在将每个数据点分配给某个聚类中心之后，算法的第二阶段会为每个聚类中心重新计算所有分配给它的数据点的均值。


### 练习 2

请补全下面的 `compute_centroids` 函数，以重新计算每个聚类中心的值：

* 具体来说，对于每个聚类中心 $\mu_k$，我们设定：
$$\mu_k = \frac{1}{|C_k|} \sum_{i \in C_k} x^{(i)}$$ 

    其中：
    * $C_k$ 是分配给聚类中心 $k$ 的样本集合
    * $|C_k|$ 是集合 $C_k$ 中样本的数量


* 具体举例，如果两个样本 $x^{(3)}$ 和 $x^{(5)}$ 被分配给了聚类中心 $k=2$，那么你应当更新 $\mu_2 = \frac{1}{2}(x^{(3)}+x^{(5)})$。

如果遇到困难，你可以查看下方单元格之后的提示来帮助你完成实现。

In [ ]:
# UNQ_C2
# GRADED FUNCTION: compute_centpods

def compute_centroids(X, idx, K):
    """
    Returns the new centroids by computing the means of the 
    data points assigned to each centroid.
    
    Args:
        X (ndarray):   (m, n) Data points
        idx (ndarray): (m,) Array containing index of closest centroid for each 
                       example in X. Concretely, idx[i] contains the index of 
                       the centroid closest to example i
        K (int):       number of centroids
    
    Returns:
        centroids (ndarray): (K, n) New centroids computed
    """
    
    # Useful variables
    m, n = X.shape
    
    # You need to return the following variables correctly
    centroids = np.zeros((K, n))
    
    ### START CODE HERE ###

    ### END CODE HERE ## 
    
    return centroids

<details>
  <summary><font size="3" color="darkgreen"><b>Click for hints</b></font></summary>
    
    
* Here's how you can structure the overall implementation for this function
    ```python 
    def compute_centroids(X, idx, K):
        # Useful variables
        m, n = X.shape
    
        # You need to return the following variables correctly
        centroids = np.zeros((K, n))
    
        ### START CODE HERE ###
        for k in range(K):   
            points = # Your code here to get a list of all data points in X assigned to centroid k  
            centroids[k] = # Your code here to compute the mean of the points assigned
    ### END CODE HERE ## 
    
    return centroids
    ```
  
    If you're still stuck, you can check the hints presented below to figure out how to calculate `points` and `centroids[k]`.
    
    <details>
          <summary><font size="2" color="darkblue"><b>Hint to calculate points</b></font></summary>
           &emsp; &emsp; Say we wanted to find all the values in X that were assigned to cluster <code>k=0</code>. That is, the corresponding value in idx for these examples is 0. In Python, we can do it as <code>X[idx == 0]</code>. Similarly, the points assigned to centroid <code>k=1</code> are <code>X[idx == 1]</code>
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; More hints to calculate points</b></font></summary>
               &emsp; &emsp; You can compute points as <code>points = X[idx == k] </code>
           </details>
    </details>

     <details>
          <summary><font size="2" color="darkblue"><b>Hint to calculate centroids[k]</b></font></summary>
          &emsp; &emsp; You can use <a href="https://numpy.org/doc/stable/reference/generated/numpy.mean.html">np.mean</a> to find the mean. Make sure to set the parameter <code>axis=0</code> 
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; More hints to calculate centroids[k]</b></font></summary>
              &emsp; &emsp; You can compute centroids[k] as <code>centroids[k] = np.mean(points, axis = 0)</code>
          </details>
    </details>
        
    </details>

</details>

    


<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
    
* 以下是该函数的整体实现结构：
    ```python 
    def compute_centroids(X, idx, K):
        # 有用的变量
        m, n = X.shape
    
        # 你需要正确返回以下变量
        centroids = np.zeros((K, n))
    
        ### START CODE HERE ###
        for k in range(K):   
            points = # 在此处编写代码，获取 X 中分配给聚类中心 k 的所有数据点列表  
            centroids[k] = # 在此处编写代码，计算这些已分配数据点的均值
    ### END CODE HERE ## 
    
    return centroids
    ```
  
    如果你仍然卡住了，可以查看下面的提示，了解如何计算 `points` 和 `centroids[k]`。
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 points 的提示</b></font></summary>
           &emsp; &emsp; 假设我们想找出 X 中所有分配给聚类 <code>k=0</code> 的值。也就是说，这些样本在 idx 中对应的值为 0。在 Python 中，我们可以写作 <code>X[idx == 0]</code>。类似地，分配给聚类中心 <code>k=1</code> 的点是 <code>X[idx == 1]</code>
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 关于计算 points 的更多提示</b></font></summary>
               &emsp; &emsp; 你可以这样计算 points：<code>points = X[idx == k] </code>
           </details>
    </details>

     <details>
          <summary><font size="2" color="darkblue"><b>计算 centroids[k] 的提示</b></font></summary>
          &emsp; &emsp; 你可以使用 <a href="https://numpy.org/doc/stable/reference/generated/numpy.mean.html">np.mean</a> 来求均值。请确保设置参数 <code>axis=0</code> 
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 关于计算 centroids[k] 的更多提示</b></font></summary>
              &emsp; &emsp; 你可以这样计算 centroids[k]：<code>centroids[k] = np.mean(points, axis = 0)</code>
          </details>
    </details>
        
    </details>

</details>

Now check your implementation by running the cell below

现在运行下面的代码单元格来检查你的实现

In [ ]:
K = 3
centroids = compute_centroids(X, idx, K)

print("The centroids are:", centroids)

# UNIT TEST
compute_centroids_test(compute_centroids)


**Expected Output**:

2.42830111 3.15792418

5.81350331 2.63365645

7.11938687 3.6166844 

**预期输出**：

2.42830111 3.15792418

5.81350331 2.63365645

7.11938687 3.6166844

<a name="2"></a>
## 2 - K-means on a sample dataset 

After you have completed the two functions (`find_closest_centroids`
and `compute_centroids`) above, the next step is to run the
K-means algorithm on a toy 2D dataset to help you understand how
K-means works. 
* We encourage you to take a look at the function (`run_kMeans`) below to understand how it works. 
* Notice that the code calls the two functions you implemented in a loop.

When you run the code below, it will produce a
visualization that steps through the progress of the algorithm at
each iteration. 
* At the end, your figure should look like the one displayed in Figure 1.

<img src="images/figure 1.png" width="500" height="500">


**Note**: You do not need to implement anything for this part. Simply run the code provided below

## 2 - 在示例数据集上运行 K-means

在完成上面的两个函数（`find_closest_centroids` 和 `compute_centroids`）之后，下一步是在一个简单的二维数据集上运行 K-means 算法，以帮助你理解 K-means 的工作原理。
* 我们鼓励你查看下面的函数（`run_kMeans`），以了解它的工作方式。
* 注意，代码在一个循环中调用了你实现的这两个函数。

当你运行下面的代码时，它会生成一个可视化图形，逐步展示算法在每次迭代中的进展。
* 最终，你的图形应该与图 1 中显示的相似。

**注意**：本部分无需你实现任何代码，只需运行下面提供的代码即可。

In [ ]:
# You do not need to implement anything for this part

def run_kMeans(X, initial_centroids, max_iters=10, plot_progress=False):
    """
    Runs the K-Means algorithm on data matrix X, where each row of X
    is a single example
    """
    
    # Initialize values
    m, n = X.shape
    K = initial_centroids.shape[0]
    centroids = initial_centroids
    previous_centroids = centroids    
    idx = np.zeros(m)
    
    # Run K-Means
    for i in range(max_iters):
        
        #Output progress
        print("K-Means iteration %d/%d" % (i, max_iters-1))
        
        # For each example in X, assign it to the closest centroid
        idx = find_closest_centroids(X, centroids)
        
        # Optionally plot progress
        if plot_progress:
            plot_progress_kMeans(X, centroids, previous_centroids, idx, K, i)
            previous_centroids = centroids
            
        # Given the memberships, compute new centroids
        centroids = compute_centroids(X, idx, K)
    plt.show() 
    return centroids, idx

In [ ]:
# Load an example dataset
X = load_data()

# Set initial centroids
initial_centroids = np.array([[3,3],[6,2],[8,5]])
K = 3

# Number of iterations
max_iters = 10

centroids, idx = run_kMeans(X, initial_centroids, max_iters, plot_progress=True)

<a name="3"></a>
## 3 - Random initialization

The initial assignments of centroids for the example dataset was designed so that you will see the same figure as in Figure 1. In practice, a good strategy for initializing the centroids is to select random examples from the
training set.

In this part of the exercise, you should understand how the function `kMeans_init_centroids` is implemented.
* The code first randomly shuffles the indices of the examples (using `np.random.permutation()`). 
* Then, it selects the first $K$ examples based on the random permutation of the indices. 
    * This allows the examples to be selected at random without the risk of selecting the same example twice.

**Note**: You do not need to make implement anything for this part of the exercise.

## 3 - 随机初始化

示例数据集的初始聚类中心分配是预先设计好的，以便你能看到与图 1 相同的图形。在实践中，初始化聚类中心的一个好策略是从训练集中随机选择样本。

在本部分练习中，你应该理解函数 `kMeans_init_centroids` 是如何实现的。
* 代码首先随机打乱样本的索引（使用 `np.random.permutation()`）。
* 然后，它根据随机排列后的索引选择前 $K$ 个样本。
    * 这样可以随机选择样本，而不会有重复选择同一个样本的风险。

**注意**：本部分练习无需你实现任何代码。

In [ ]:
# You do not need to modify this part

def kMeans_init_centroids(X, K):
    """
    This function initializes K centroids that are to be 
    used in K-Means on the dataset X
    
    Args:
        X (ndarray): Data points 
        K (int):     number of centroids/clusters
    
    Returns:
        centroids (ndarray): Initialized centroids
    """
    
    # Randomly reorder the indices of examples
    randidx = np.random.permutation(X.shape[0])
    
    # Take the first K examples as centroids
    centroids = X[randidx[:K]]
    
    return centroids

<a name="4"></a>
## 4 - Image compression with K-means

In this exercise, you will apply K-means to image compression. 

* In a straightforward 24-bit color representation of an image$^{2}$, each pixel is represented as three 8-bit unsigned integers (ranging from 0 to 255) that specify the red, green and blue intensity values. This encoding is often refered to as the RGB encoding.
* Our image contains thousands of colors, and in this part of the exercise, you will reduce the number of
colors to 16 colors.
* By making this reduction, it is possible to represent (compress) the photo in an efficient way. 
* Specifically, you only need to store the RGB values of the 16 selected colors, and for each pixel in the image you now need to only store the index of the color at that location (where only 4 bits are necessary to represent 16 possibilities).

In this part, you will use the K-means algorithm to select the 16 colors that will be used to represent the compressed image.
* Concretely, you will treat every pixel in the original image as a data example and use the K-means algorithm to find the 16 colors that best group (cluster) the pixels in the 3- dimensional RGB space. 
* Once you have computed the cluster centroids on the image, you will then use the 16 colors to replace the pixels in the original image.

<img src="images/figure 2.png" width="500" height="500">

$^{2}$<sub>The provided photo used in this exercise belongs to Frank Wouters and is used with his permission.</sub>

<a name="4.1"></a>
### 4.1 Dataset

**Load image**

First, you will use `matplotlib` to read in the original image, as shown below.

## 4 - 使用 K-means 进行图像压缩

在本练习中，你将把 K-means 应用于图像压缩。

* 在直观的 24 位彩色图像表示中$^{2}$，每个像素由三个 8 位无符号整数（范围从 0 到 255）表示，分别指定红、绿、蓝的强度值。这种编码通常称为 RGB 编码。
* 我们的图像包含数千种颜色，在本部分练习中，你将把颜色数量减少到 16 种。
* 通过这种缩减，可以以高效的方式表示（压缩）这张照片。
* 具体来说，你只需要存储所选的 16 种颜色的 RGB 值，而对于图像中的每个像素，现在只需要存储该位置颜色的索引（只需 4 位即可表示 16 种可能）。

在本部分中，你将使用 K-means 算法选择用于表示压缩图像的 16 种颜色。
* 具体而言，你将把原始图像中的每个像素视为一个数据样本，并使用 K-means 算法在三维 RGB 空间中找出最能对像素进行分组（聚类）的 16 种颜色。
* 计算出图像上的聚类中心后，你将使用这 16 种颜色替换原始图像中的像素。

$^{2}$<sub>本练习中使用的照片归 Frank Wouters 所有，经其许可使用。</sub>

### 4.1 数据集

**加载图像**

首先，你将使用 `matplotlib` 读取原始图像，如下所示。

In [ ]:
# Load an image of a bird
original_img = plt.imread('bird_small.png')

**Visualize image**

You can visualize the image that was just loaded using the code below.

**可视化图像**

你可以使用下面的代码可视化刚刚加载的图像。

In [ ]:
# Visualizing the image
plt.imshow(original_img)

**Check the dimension of the variable**

As always, you will print out the shape of your variable to get more familiar with the data.

**检查变量的维度**

与往常一样，你将打印出变量的形状，以便更熟悉数据。

In [ ]:
print("Shape of original_img is:", original_img.shape)

As you can see, this creates a three-dimensional matrix `original_img` where 
* the first two indices identify a pixel position, and
* the third index represents red, green, or blue. 

For example, `original_img[50, 33, 2]` gives the blue intensity of the pixel at row 50 and column 33.

#### Processing data

To call the `run_kMeans`, you need to first transform the matrix `original_img` into a two-dimensional matrix.

* The code below reshapes the matrix `original_img` to create an $m \times 3$ matrix of pixel colors (where
$m=16384 = 128\times128$)

如你所见，这会创建一个三维矩阵 `original_img`，其中
* 前两个索引标识像素的位置，
* 第三个索引表示红、绿或蓝通道。

例如，`original_img[50, 33, 2]` 给出第 50 行、第 33 列像素的蓝色通道强度。

#### 处理数据

要调用 `run_kMeans`，你需要先将矩阵 `original_img` 转换为二维矩阵。

* 下面的代码将矩阵 `original_img` 重塑为一个 $m \times 3$ 的像素颜色矩阵（其中 $m=16384 = 128\times128$）。

In [ ]:
# Divide by 255 so that all values are in the range 0 - 1
original_img = original_img / 255

# Reshape the image into an m x 3 matrix where m = number of pixels
# (in this case m = 128 x 128 = 16384)
# Each row will contain the Red, Green and Blue pixel values
# This gives us our dataset matrix X_img that we will use K-Means on.

X_img = np.reshape(original_img, (original_img.shape[0] * original_img.shape[1], 3))

<a name="4.2"></a>
### 4.2 K-Means on image pixels

Now, run the cell below to run K-Means on the pre-processed image.

### 4.2 对图像像素运行 K-Means

现在，运行下面的单元格对预处理后的图像运行 K-Means。

In [ ]:
# Run your K-Means algorithm on this data
# You should try different values of K and max_iters here
K = 16                       
max_iters = 10               

# Using the function you have implemented above. 
initial_centroids = kMeans_init_centroids(X_img, K) 

# Run K-Means - this takes a couple of minutes
centroids, idx = run_kMeans(X_img, initial_centroids, max_iters) 

In [ ]:
print("Shape of idx:", idx.shape)
print("Closest centroid for the first five elements:", idx[:5])

<a name="4.3"></a>
### 4.3 Compress the image


### 4.3 压缩图像

After finding the top $K=16$ colors to represent the image, you can now
assign each pixel position to its closest centroid using the
`find_closest_centroids` function. 
* This allows you to represent the original image using the centroid assignments of each pixel. 
* Notice that you have significantly reduced the number of bits that are required to describe the image. 
    * The original image required 24 bits for each one of the $128\times128$ pixel locations, resulting in total size of $128 \times 128 \times 24 = 393,216$ bits. 
    * The new representation requires some overhead storage in form of a dictionary of 16 colors, each of which require 24 bits, but the image itself then only requires 4 bits per pixel location. 
    * The final number of bits used is therefore $16 \times 24 + 128 \times 128 \times 4 = 65,920$ bits, which corresponds to compressing the original image by about a factor of 6.

在找到代表图像的前 $K=16$ 种颜色后，你现在可以使用 `find_closest_centroids` 函数将每个像素位置分配给其最近的聚类中心。
* 这使你能够仅用每个像素的聚类中心分配来表示原始图像。
* 请注意，你已大幅减少了描述该图像所需的比特数。
    * 原始图像为 $128\times128$ 个像素位置中的每一个都需要 24 位，总大小为 $128 \times 128 \times 24 = 393,216$ 位。
    * 新的表示方法需要以 16 种颜色字典的形式占用少量额外存储开销，每种颜色需要 24 位，但图像本身每个像素位置只需要 4 位。
    * 因此，最终使用的总比特数为 $16 \times 24 + 128 \times 128 \times 4 = 65,920$ 位，相当于将原始图像压缩了约 6 倍。

In [ ]:
# Represent image in terms of indices
X_recovered = centroids[idx, :] 

# Reshape recovered image into proper dimensions
X_recovered = np.reshape(X_recovered, original_img.shape) 

Finally, you can view the effects of the compression by reconstructing
the image based only on the centroid assignments. 
* Specifically, you can replace each pixel location with the mean of the centroid assigned to
it. 
* Figure 3 shows the reconstruction we obtained. Even though the resulting image retains most of the characteristics of the original, we also see some compression artifacts.

<img src="images/figure 3.png" width="700" height="700">


最后，你可以仅根据聚类中心分配来重构图像，从而查看压缩效果。
* 具体来说，你可以用分配给每个像素位置的聚类中心均值来替换该像素。
* 图 3 显示了我们获得的重构结果。尽管生成的图像保留了原始图像的大部分特征，但我们也能看到一些压缩伪影。

In [ ]:
# Display original image
fig, ax = plt.subplots(1,2, figsize=(8,8))
plt.axis('off')

ax[0].imshow(original_img*255)
ax[0].set_title('Original')
ax[0].set_axis_off()


# Display compressed image
ax[1].imshow(X_recovered*255)
ax[1].set_title('Compressed with %d colours'%K)
ax[1].set_axis_off()